# BGE-M3 Embedding Batch Size Benchmark

根据需求，本 Notebook 旨在通过真实数据测算本地 Ollama 及 GPU 推理过程中最合适的 Embedding Batch Size 配置。

### 测试要求与设计
1. **前提**：关闭 Ollama，避免显存测量失真。
2. **数据**：从 ESCI 取 512 条，拼接标题与描述，避免使用等长的假字符串，基于真实长度分布测算，确保最长文本不出错。
3. **翻倍探顶**：32 起步翻倍，测峰值显存与吞吐，直到 OOM。
4. **回退留量**：遇 OOM 则回退一档并预留 20% 以防显存碎片化崩溃。
5. **识别拐点**：注意吞吐会先见顶。如果吞吐见顶而显存翻倍，优先取吞吐拐点（即上限是显存约束，最优值是吞吐曲线的拐点）。
6. **耗时估算**：换算周日预期任务耗时（乘 1.5–2 倍 I/O 余量）。若超 90 分钟则输出警报。
7. **输出报告**：生成可供贴入 `docs/DECISIONS.md` 以及工程指标表的内容。

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

# --- 1. 配置参数 ---
DATA_PATH = "../data/raw/shopping_queries_dataset_products.parquet"
MODEL_NAME = "BAAI/bge-m3"
MAX_LEN = 512
BATCH_SIZES = [32, 64, 128, 256, 512]
REPS = 3
TOTAL_PRODUCTS = 50_000  # 用于预估周末入库任务的总量

d:\Github_Clones\RetrievalCore\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- 2. 检查 GPU 是否空闲 ---
def assert_gpu_is_free(limit_mib: int = 500) -> None:
    if not torch.cuda.is_available():
        raise SystemExit("没有检测到可用的 CUDA 设备")
    
    free, total = torch.cuda.mem_get_info()
    used = (total - free) / 1024**2
    
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")
    print(f"显存状态: 总量 {total / 1024**3:.1f} GiB, 已使用 {used:.0f} MiB\n")
    
    if used > limit_mib:
        raise SystemExit(
            "⚠️ GPU 不空闲。请运行 `ollama stop <model>` (或停止其后台服务) 后重试。\n"
            "如果有 Ollama 等其他进程常驻后台，测得的显存天花板会偏低，导致测试无效。"
        )

assert_gpu_is_free()

GPU 设备: NVIDIA GeForce RTX 2060 SUPER
显存状态: 总量 8.0 GiB, 已使用 1034 MiB



SystemExit: ⚠️ GPU 不空闲。请运行 `ollama stop <model>` (或停止其后台服务) 后重试。
如果有 Ollama 等其他进程常驻后台，测得的显存天花板会偏低，导致测试无效。

d:\Github_Clones\RetrievalCore\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# --- 3. 构造代表性文本 (标题+描述) ---
def load_and_concat_texts(path: str, sample_size: int = 512) -> list:
    print(f"正在从 {path} 读取数据...")
    df = pd.read_parquet(path)
    
    # 拼接商品标题与描述
    df['product_title'] = df['product_title'].fillna('').astype(str)
    df['product_description'] = df['product_description'].fillna('').astype(str)
    combined = (df['product_title'] + " " + df['product_description']).str.strip()
    
    # 过滤空内容
    texts = combined[combined != ""].tolist()
    if not texts:
        raise ValueError("清洗后没有可用文本，请检查数据。")
        
    # 随手取指定数量（维持真实长度分布，让最长的那条决定真实的显存占用）
    sampled = texts[:sample_size]
    
    lens = sorted(len(t) for t in sampled)
    print(f"已抽取 {len(sampled)} 条代表性文本。")
    print(f"字符长度分布: P50={lens[len(lens)//2]}, P95={lens[int(len(lens)*0.95)]}, Max={lens[-1]}")
    
    return sampled

base_texts = load_and_concat_texts(DATA_PATH, sample_size=512)

正在从 ../data/raw/shopping_queries_dataset_products.parquet 读取数据...
已抽取 512 条代表性文本。
字符长度分布: P50=264, P95=1903, Max=2691


In [ ]:
# --- 4. 探顶压测核心逻辑 ---
@torch.inference_mode()
def encode(model, tok, batch: list, max_len: int) -> torch.Tensor:
    enc = tok(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    enc = {k: v.to("cuda") for k, v in enc.items()}
    cls = model(**enc).last_hidden_state[:, 0]
    return F.normalize(cls, dim=-1)

def bench(model, tok, pool: list, bs: int, max_len: int, reps: int) -> tuple:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    # 从样本池中循环取文本，保证不同 batch 处理时的长度特征一致
    batches = []
    for i in range(reps):
        batch = [pool[(i * bs + j) % len(pool)] for j in range(bs)]
        batches.append(batch)

    # 预热 (不计入时间)
    encode(model, tok, batches[0], max_len)  
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for b in batches:
        encode(model, tok, b, max_len)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0

    peak_gib = torch.cuda.max_memory_reserved() / 1024**3
    return (bs * reps) / dt, peak_gib

In [ ]:
# --- 5. 执行翻倍探顶 ---
print(f"加载模型 {MODEL_NAME} ...")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to("cuda").eval()

results = []
print(f"\n{'Batch':>6} {'条/秒':>9} {'峰值(GiB)':>9}")
print("-" * 27)

for bs in BATCH_SIZES:
    try:
        tp, peak = bench(model, tok, base_texts, bs, MAX_LEN, REPS)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
        if "out of memory" not in str(exc).lower():
            raise
        print(f"{bs:>6} {'OOM':>9}")
        torch.cuda.empty_cache()
        break
        
    results.append({"batch_size": bs, "items_per_s": round(tp, 1), "peak_gib": round(peak, 2)})
    print(f"{bs:>6} {tp:>9.1f} {peak:>9.2f}")

if not results:
    raise SystemExit("\n最小的 Batch (32) 也发生 OOM，请减小 max_len 截断长度。")

加载模型 BAAI/bge-m3 ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2657.61it/s]



 Batch       条/秒   峰值(GiB)
---------------------------
    32      88.7      1.74
    64      71.0      2.24
   128      70.4      3.33
   256      69.6      5.58
   512      13.7     10.10


In [ ]:
# --- 6. 回退与留余量，以及 ETA 估算 ---
best_tp = max(r["items_per_s"] for r in results)

# 寻找吞吐曲线拐点 (定义为达到最大吞吐 95% 的最小 batch size)
knee_result = next(r for r in results if r["items_per_s"] >= 0.95 * best_tp)
largest_result = results[-1]

# 判断：是否在吞吐见顶前就撞到了显存天花板 (即未 OOM 之前的最后一档仍然没有达到边际收益平台期)
hit_memory_wall_before_plateau = (knee_result["batch_size"] == largest_result["batch_size"]) and (len(results) < len(BATCH_SIZES))

chosen = knee_result["batch_size"]
if hit_memory_wall_before_plateau:
    # 回退并留余量: 取 80% 作为最终值，防止彻夜长跑遇到真实长文本时崩溃
    chosen = max(8, int(chosen * 0.8) // 8 * 8)
    note = "撞到显存墙前吞吐量未见顶；为保证彻夜执行的安全，已回退并留出 20% 余量。"
else:
    # 吞吐量先见顶（拐点早于OOM），此时提升 Batch 只会增加显存消耗但毫无性能收益
    note = "吞吐量已达拐点（边际收益极低），取拐点值，本身已拥有天然的显存余量。"

tp = next((r["items_per_s"] for r in results if r["batch_size"] == chosen), knee_result["items_per_s"])

# 换算周末预估耗时 (含 I/O 与入库余量)
pure_min = TOTAL_PRODUCTS / tp / 60
total_min_est_lower = pure_min * 1.5
total_min_est_upper = pure_min * 2.0

print("\n=== 最终推荐配置 ===")
print(f"推荐 Batch Size : {chosen}  ({note})")
print(f"预期吞吐量      : {tp:.1f} 条/秒")
print(f"峰值显存占用    : {knee_result['peak_gib']:.2f} GiB (压测最高峰值)")

print(f"\n=== 周日入库耗时预估 (基于 {TOTAL_PRODUCTS:,} 条数据) ===")
print(f"纯推理耗时      : {pure_min:.0f} 分钟")
print(f"总耗时 (含I/O)  : 约 {total_min_est_lower:.0f} - {total_min_est_upper:.0f} 分钟")

if total_min_est_upper > 90:
    print("\n⚠️ 警告: 预估总耗时超过 90 分钟！")
    print("建议行动: 要么考虑降低抽样规模，要么周六晚上提前把 embedding 任务挂上。")


=== 最终推荐配置 ===
推荐 Batch Size : 32  (吞吐量已达拐点（边际收益极低），取拐点值，本身已拥有天然的显存余量。)
预期吞吐量      : 88.7 条/秒
峰值显存占用    : 1.74 GiB (压测最高峰值)

=== 周日入库耗时预估 (基于 50,000 条数据) ===
纯推理耗时      : 9 分钟
总耗时 (含I/O)  : 约 14 - 19 分钟


In [ ]:
# --- 7. 输出报告 (入库 docs/DECISIONS.md) ---
date_str = time.strftime('%Y-%m-%d')
device_name = torch.cuda.get_device_name(0)

markdown_entry = f"""### {date_str} - BGE-M3 Embedding Batch Size Configuration

* **Batch Size**: `{chosen}`
* **Peak Memory**: `{knee_result['peak_gib']:.2f} GiB`
* **Throughput**: `{tp:.0f} items/s`
* **ETA**: `~{total_min_est_upper:.0f} min` for {TOTAL_PRODUCTS:,} products (including I/O & Upsert overhead)

**Notes**: Measured on {device_name} with `max_len={MAX_LEN}` using fp16. {note}
"""

print("--- 请将以下内容复制并粘贴到 docs/DECISIONS.md 以及 README 的工程指标表中 ---\n")
print(markdown_entry)

# 保存测试记录
out_path = Path("../docs/bench_batch_size.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps({
    "date": date_str,
    "device": device_name,
    "max_len": MAX_LEN,
    "total_products": TOTAL_PRODUCTS,
    "results": results,
    "chosen": chosen,
    "note": note
}, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nJSON 测试结果已归档至: {out_path}")

--- 请将以下内容复制并粘贴到 docs/DECISIONS.md 以及 README 的工程指标表中 ---

### 2026-08-30 - BGE-M3 Embedding Batch Size Configuration

* **Batch Size**: `32`
* **Peak Memory**: `1.74 GiB`
* **Throughput**: `89 items/s`
* **ETA**: `~19 min` for 50,000 products (including I/O & Upsert overhead)

**Notes**: Measured on NVIDIA GeForce RTX 2060 SUPER with `max_len=512` using fp16. 吞吐量已达拐点（边际收益极低），取拐点值，本身已拥有天然的显存余量。


JSON 测试结果已归档至: docs\bench_batch_size.json
